# `ncvoter` — profile, choose columns & trim

Trim type: **column**. Choose **11 columns**, **35,000 rows**. Save `ncvoter_11c_35000r.csv`.

> Output columns: `c0`…`c10`.

In [1]:
import os
import numpy as np
import pandas as pd

NAME       = "ncvoter"
N_ROWS     = 35000
N_COLS     = 11
TRIM_ROWS  = "sample"   # head | sample

RAW_PATH   = "ncvoter_22_1m.csv"
OUT_PATH   = f"{NAME}_{N_COLS}c_{N_ROWS}r.csv"
DELIM      = ","
OUT_DELIM  = DELIM
HAS_HEADER = True
ENCODING   = "latin-1"
ON_BAD_LINES = None

## 1. View the raw data

In [2]:
read_kw = dict(sep=DELIM, header=0 if HAS_HEADER else None, encoding=ENCODING, low_memory=False)
if ON_BAD_LINES is not None:
    read_kw["on_bad_lines"] = ON_BAD_LINES
raw = pd.read_csv(RAW_PATH, **read_kw)
raw.columns = [f"c{i}" for i in range(raw.shape[1])]
print("raw shape :", raw.shape)
print("columns   :", list(raw.columns))
raw.head()

raw shape : (938084, 22)
columns   : ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15', 'c16', 'c17', 'c18', 'c19', 'c20', 'c21']


,c0,c1,c2,c3,c4,c5,c6,c7,c8,c9,...,c12,c13,c14,c15,c16,c17,c18,c19,c20,c21
0,1,ALAMANCE,9048723,A,ACTIVE,AV,VERIFIED,"AARON, CHRISTINA CASTAGNA",CHRISTINA CASTAGNA AARON,421,...,UN,UNDESIGNATED,UNA,UNAFFILIATED,F,FEMALE,36,03/26/1996,Age 26 - 40,AA98377
1,1,ALAMANCE,9019674,A,ACTIVE,AV,VERIFIED,"AARON, CLAUDIA HAYDEN",CLAUDIA HAYDEN AARON,1013,...,NL,NOT HISPANIC or NOT LATINO,UNA,UNAFFILIATED,F,FEMALE,67,08/15/1989,Age Over 66,AA69747
2,1,ALAMANCE,9129589,A,ACTIVE,AV,VERIFIED,"AARON, JAMES MICHAEL",JAMES MICHAEL AARON,1647,...,UN,UNDESIGNATED,DEM,DEMOCRATIC,M,MALE,64,03/07/2012,Age 41 - 65,AA170513
3,1,ALAMANCE,9041748,A,ACTIVE,AV,VERIFIED,"AARON, NATHAN EDWARD",NATHAN EDWARD AARON,421,...,UN,UNDESIGNATED,UNA,UNAFFILIATED,M,MALE,36,10/10/1994,Age 26 - 40,AA91549
4,1,ALAMANCE,9021947,A,ACTIVE,AV,VERIFIED,"AARON, WILLIE DALE",WILLIE DALE AARON,1013,...,NL,NOT HISPANIC or NOT LATINO,UNA,UNAFFILIATED,M,MALE,68,06/06/1990,Age Over 66,AA71983


In [ ]:
raw.dtypes

## 2. Profile: cardinality, top-value %, and group skew

In [ ]:
total_rows = len(raw)

rows = []
for col in raw.columns:
    vc = raw[col].value_counts(dropna=False)
    distinct = int(raw[col].nunique(dropna=True))
    nulls    = int(raw[col].isna().sum() + (raw[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))

prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")

prof

## 3. Choose columns (cardinality mix + id/super-key)

In [ ]:
prof_sel = prof[prof.distinct_values > 1]
KEY_THRESH = 95
keys     = prof_sel[prof_sel.unique_percent >= KEY_THRESH]
non_keys = prof_sel[prof_sel.unique_percent <  KEY_THRESH]
chosen = []
if len(keys):
    chosen.append(keys.sort_values("unique_percent", ascending=False).index[0])
n_left = N_COLS - len(chosen)
nk     = non_keys.sort_values("unique_percent")
ix_strata = np.array_split(nk.index.to_numpy(), min(3, max(1, len(nk)))) if len(nk) else []
strata = [nk.loc[ix].sort_values("skew_ratio", ascending=False) for ix in ix_strata]
ptr    = [0] * len(strata)
while n_left > 0 and any(ptr[i] < len(strata[i]) for i in range(len(strata))):
    for i in range(len(strata)):
        if ptr[i] < len(strata[i]):
            chosen.append(strata[i].index[ptr[i]]); ptr[i] += 1; n_left -= 1
            if n_left == 0: break
for c in raw.columns:
    if len(chosen) >= N_COLS: break
    if c not in chosen: chosen.append(c)
SELECTED_COLS = [c for c in raw.columns if c in set(chosen)][:N_COLS]
assert len(SELECTED_COLS) == N_COLS, SELECTED_COLS
print("selected columns:", SELECTED_COLS)

## 4. Trim to the chosen columns x exact rows

In [ ]:
assert raw.shape[1] >= N_COLS, f"need >= {N_COLS} cols, have {raw.shape[1]}"
n_take = min(N_ROWS, len(raw))
assert len(raw) >= n_take, f"need >= {n_take} rows, have {len(raw)}"

if TRIM_ROWS == "sample":
    trimmed = raw.loc[:, SELECTED_COLS].sample(n_take, random_state=42).reset_index(drop=True)
else:
    trimmed = raw.loc[:, SELECTED_COLS].iloc[:n_take].copy()
trimmed.columns = [f"c{i}" for i in range(N_COLS)]

assert trimmed.shape == (n_take, N_COLS), trimmed.shape
print("trimmed shape:", trimmed.shape)
trimmed.head()

## 5. Save the trimmed CSV

In [ ]:
out_dir = os.path.dirname(OUT_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)
trimmed.to_csv(OUT_PATH, index=False, sep=OUT_DELIM)
print("wrote", OUT_PATH, trimmed.shape)

chk = pd.read_csv(OUT_PATH, sep=OUT_DELIM)
print("reloaded:", chk.shape)
print("columns   :", list(chk.columns))

## 6. Check selected cardinality and skew

In [ ]:
output = pd.read_csv(OUT_PATH, sep=OUT_DELIM, encoding=ENCODING, low_memory=False)
print("output shape :", output.shape)
print("columns      :", list(output.columns))
output.head()

In [ ]:
total_rows = len(output)
rows = []
for col in output.columns:
    vc = output[col].value_counts(dropna=False)
    distinct = int(output[col].nunique(dropna=True))
    nulls    = int(output[col].isna().sum() + (output[col].astype(str) == "").sum())
    rows.append((
        col, distinct, nulls,
        round(nulls    / total_rows * 100, 2),
        round(distinct / total_rows * 100, 2),
        round(vc.iloc[0] / total_rows * 100, 2),
        round(vc.mean(), 2), int(vc.max()),
        round(vc.max() / vc.mean(), 2),
    ))
out_prof = pd.DataFrame(rows, columns=[
    "column", "distinct_values", "null_count", "null_percent",
    "unique_percent", "top_value_percent", "mean_group", "max_group", "skew_ratio",
]).set_index("column")
out_prof